# Arbeitspaket (AP) 7: Management & Nutzung relationaler Daten

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[TT.MM.2026] / Raum: [anpassen] / [hh:mm – hh:mm]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung, Datenqualität &amp; NULL-Werte</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (hotel.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (hotel.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (z.B. Problem richtig benannt, aber falsch behoben; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

Die **fiktive** Schweizer Hotelkette «AlpenStay» betreibt sechs Hotels. Die Datenbank enthält Zimmer, Mitarbeitende, Gäste, Buchungskanäle und die Buchungen der Jahre 2024–2025.

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `hotel.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — nur zu verwenden, falls die PostgreSQL-Verbindung nicht funktioniert.

**Tabellen:** `hotels`, `rooms`, `employees`, `guests`, `channels`, `bookings`

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `hotel_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [1]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

Working directory: /home/claude/out/DS_HS2026_LNW_I_Examples/AP07


### Datenbankverbindung zu PostgreSQL

In [2]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:      ', host)
    print('Datenbank: ', connection.execute(text('SELECT current_database()')).scalar())
engine.dispose()

Host:       localhost
Datenbank:  postgres


### Daten nach PostgreSQL laden

In [3]:
# Tabellen aus der SQLite-Datei hotel.db nach PostgreSQL übertragen
tables = ['hotels', 'rooms', 'employees', 'guests', 'channels', 'bookings']

sqlite_engine = create_engine('sqlite:///hotel.db')
engine = create_engine(db_connection_url)

for table in tables:
    df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(f'{table:<12}{len(df):>5} Zeilen')

sqlite_engine.dispose()
engine.dispose()

hotels          6 Zeilen
rooms         112 Zeilen
employees      45 Zeilen
guests        614 Zeilen
channels        4 Zeilen
bookings     2500 Zeilen


> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für alle Aufgaben die
> SQLite-Datenbank `hotel.db`. Ersetzen Sie dazu in den Code-Zellen:
> ```python
> engine = create_engine(db_connection_url)
> ```
> durch:
>
> ```python
> engine = create_engine('sqlite:///hotel.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Prüfung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `hotel.db` werden nicht gewertet.*

### A1 (4 Punkte)

Betrachten Sie das ER-Diagramm von `hotel.db`.

a) Welche **Kardinalität** hat die Beziehung zwischen `hotels` und `rooms`, und welche zwischen `guests` und `rooms`? Begründen Sie mit dem Hotelbetrieb.

b) Wie wird die Beziehung zwischen `guests` und `rooms` im relationalen Modell umgesetzt? Warum gehören die Spalten `nights`, `rate_paid` und `status` genau dorthin — und nicht in `guests` oder `rooms`?

**Ihre Antwort:**

a) `hotels` – `rooms`: **1 : n** — ein Hotel hat viele Zimmer, jedes Zimmer gehört zu genau einem Hotel (Fremdschlüssel `rooms.hotelid`).
`guests` – `rooms`: **m : n** — ein Gast kann über die Zeit viele Zimmer buchen, und ein Zimmer wird von vielen verschiedenen Gästen gebucht.

b) Eine m:n-Beziehung lässt sich nicht mit einem einzelnen Fremdschlüssel abbilden. Sie wird über die **Zwischentabelle** (assoziative Entität) `bookings` aufgelöst, die zwei 1:n-Beziehungen enthält: `bookings.guestid → guests` und `bookings.roomid → rooms`. `nights`, `rate_paid` und `status` beschreiben **die einzelne Buchung** — also die Beziehung zwischen einem bestimmten Gast und einem bestimmten Zimmer zu einem bestimmten Zeitpunkt. In `guests` gäbe es für einen Gast mit mehreren Aufenthalten keinen eindeutigen Wert, in `rooms` ebenso wenig (das Zimmer hat nur einen Listenpreis `base_rate`).

> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Bezug: Folien Woche 1, S. 25 (Entities, Attributes, Relationships: one-to-one, one-to-many, many-to-many) und S. 27 (Relationship Cardinalities); vgl. `tickets` in `cinema.db` (m:n zwischen `customers` und `screenings`).*

### A2 (4 Punkte)

In der Tabelle `guests` fallen unter anderem folgende Einträge auf:

```
guestid  firstname  lastname  email                      birthdate   loyalty_level
G0085    Elias      Weber     elias.weber424@example.ch  1993-12-08  Gold
G0613    Elias      Weber     ELIAS.WEBER424@EXAMPLE.CH  1993-12-08  Basic
```

a) Beschreiben Sie das Datenproblem und zwei konkrete Folgen für Auswertungen oder den Betrieb von AlpenStay.

b) Mit welcher Massnahme im **Datenbankschema** hätte man das Problem verhindern können — und warum hätte ein einfacher `UNIQUE`-Constraint auf `email` hier **nicht** genügt?

**Ihre Antwort:**

a) Dieselbe Person ist **zweimal** als Gast erfasst (Dublette in den Stammdaten): Name, Geburtsdatum und E-Mail stimmen überein, nur die Schreibweise der E-Mail und die `guestid` unterscheiden sich. Folgen: (1) Auswertungen pro Gast sind falsch — z.B. zählt die Anzahl Gäste eine Person doppelt, und Umsatz bzw. Aufenthalte einer Person verteilen sich auf zwei IDs. (2) Die Stammdaten sind **inkonsistent**: Elias Weber ist einmal Gold-, einmal Basic-Mitglied — je nach Buchungsweg erhält er unterschiedliche Leistungen; zudem erhält er Werbung doppelt.

b) Ein **Eindeutigkeits-Constraint** auf der E-Mail-Adresse (bzw. auf einer Kombination wie Name + Geburtsdatum). Ein einfacher `UNIQUE`-Constraint auf `email` hätte nicht gegriffen, weil `elias.weber424@example.ch` und `ELIAS.WEBER424@EXAMPLE.CH` als Text **verschieden** sind (Vergleich unterscheidet Gross-/Kleinschreibung). Man muss die E-Mail vor dem Speichern vereinheitlichen (z.B. in Kleinbuchstaben) bzw. die Eindeutigkeit auf `LOWER(email)` prüfen. Organisatorisch hilft **Master Data Management** (eine «Golden Record» pro Gast).

> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Kontrolle: `SELECT LOWER(email), COUNT(*) FROM guests GROUP BY LOWER(email) HAVING COUNT(*) > 1` findet 4 Dubletten (G0611–G0614).*
>
> *Bezug: Folien Woche 1, S. 25 (Data Integrity Constraints: unique constraints), S. 43 (Data Quality: consistent, accurate) und S. 44 (Master Data Management).*

### A3 (4 Punkte)

Zwei Analysten zählen die Gäste in `hotel.db`:

```sql
SELECT COUNT(*) FROM guests;                -- Ergebnis: 614
SELECT COUNT(loyalty_level) FROM guests;    -- Ergebnis: 590
```

a) Warum unterscheiden sich die Ergebnisse? Welche Abfrage beantwortet die Frage «Wie viele Gäste sind erfasst?»

b) Eine dritte Abfrage `SELECT COUNT(*) FROM guests WHERE loyalty_level <> 'Gold'` liefert 532. Es gibt 58 Gäste mit `Gold`. Erklären Sie, warum das Ergebnis nicht 614 − 58 = 556 ist.

**Ihre Antwort:**

a) `COUNT(*)` zählt **alle Zeilen**, `COUNT(loyalty_level)` nur die Zeilen, in denen `loyalty_level` **nicht NULL** ist. 24 Gäste haben keinen Wert in `loyalty_level`. Die Frage «Wie viele Gäste sind erfasst?» beantwortet `COUNT(*)` (614) — oder `COUNT(guestid)`, weil der Primärschlüssel nie NULL ist.

b) Ein Vergleich mit NULL (`NULL <> 'Gold'`) ergibt in SQL weder wahr noch falsch, sondern **unbekannt** (NULL). `WHERE` behält nur Zeilen, für die die Bedingung wahr ist — die 24 Gäste ohne Stufe fallen deshalb ebenfalls weg: 614 − 58 − 24 = **532**. Will man sie mitzählen, braucht es `OR loyalty_level IS NULL` (vgl. C1).

> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Bezug: Folien Woche 1, S. 38–39 (Data Types, SQL-Abfragen); AP01 C1 (Hinweis: «`NOT IN` verliert keine Zeilen, weil `membership_type` keine NULL-Werte enthält»).*

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `hotel.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Logiserlös (`revenue`) pro Hotel in CHF für die Jahre 2024–2025, gerundet auf 2 Dezimalstellen und absteigend sortiert. Der Logiserlös ist der tatsächlich bezahlte Betrag für alle Übernachtungen aus **bestätigten** Buchungen. Hotels ohne Buchungen müssen nicht erscheinen.

```sql
SELECT h.name,
       ROUND(CAST(SUM(b.rate_paid) AS NUMERIC), 2) AS revenue
FROM hotels h
JOIN rooms r    ON h.hotelid = r.hotelid
JOIN bookings b ON r.roomid  = b.roomid
GROUP BY h.name
ORDER BY revenue DESC;
```

Tatsächliches Ergebnis der SQL-Abfrage:

```
                    name   revenue
0      AlpenStay Zermatt  229608.0
1       AlpenStay Luzern  130543.0
2  AlpenStay Grindelwald  121821.0
3         AlpenStay Bern   93470.0
4        AlpenStay Basel   92526.0
```

Die Buchhaltung meldet für AlpenStay Zermatt einen Logiserlös von rund **CHF 550'000**. Beachten Sie die Spaltenbeschreibungen im Sheet «Description».

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

1. **Preis pro Nacht × Anzahl Nächte:** `rate_paid` ist laut Sheet «Description» der bezahlte Preis **pro Nacht**. `SUM(b.rate_paid)` addiert deshalb nur eine Nacht pro Buchung. Der Betrag einer Buchung ist `b.nights * b.rate_paid` — summiert wird `SUM(b.nights * b.rate_paid)`.
2. **Stornierte Buchungen ausschliessen:** Buchungen mit `status = 'cancelled'` wurden nicht bezahlt und gehören nicht zum Logiserlös. `WHERE b.status = 'confirmed'` filtert sie vor dem Gruppieren heraus.

Ergebnis: AlpenStay **Zermatt CHF 552'446.50** (passt zur Buchhaltung), gefolgt von Luzern (CHF 340'711.50) und Grindelwald (CHF 339'119.00). Ohne die zweite Änderung wäre der Erlös von Zermatt um rund CHF 102'000 zu hoch (CHF 654'745.50).

**Korrigierte SQL-Abfrage:**

In [4]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
SELECT
    h.name,
    ROUND(CAST(SUM(b.nights * b.rate_paid) AS NUMERIC), 2) AS revenue
FROM hotels h
JOIN rooms r    ON h.hotelid = r.hotelid
JOIN bookings b ON r.roomid  = b.roomid
WHERE b.status = 'confirmed'
GROUP BY h.name
ORDER BY revenue DESC
""", con=engine)

engine.dispose()
df_B1

,name,revenue
0,AlpenStay Zermatt,552446.5
1,AlpenStay Luzern,340711.5
2,AlpenStay Grindelwald,339119.0
3,AlpenStay Bern,256079.0
4,AlpenStay Basel,255757.0


> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Bezug: Folien Woche 1, S. 39 (SQL-Abfrage mit Berechnung, `WHERE`, `GROUP BY`); Sheet «Description» von `hotel_data.xlsx`; analog AP01 A1 / AP04 B1 (bezahlter Preis statt Listenpreis).*

### B2 (8 Punkte)

**Ziel:** Anzahl **bestätigter** Buchungen (`confirmed_bookings`) pro Hotel — für **alle** Hotels. Ein Hotel ohne bestätigte Buchungen soll mit `0` erscheinen.

```sql
SELECT h.name,
       COUNT(b.bookingid) AS confirmed_bookings
FROM hotels h
LEFT JOIN rooms r    ON h.hotelid = r.hotelid
LEFT JOIN bookings b ON r.roomid  = b.roomid
WHERE b.status = 'confirmed'
GROUP BY h.name
ORDER BY confirmed_bookings DESC;
```

Tatsächliches Ergebnis:

```
                    name  confirmed_bookings
0         AlpenStay Bern                 469
1  AlpenStay Grindelwald                 462
2       AlpenStay Luzern                 428
3      AlpenStay Zermatt                 423
4        AlpenStay Basel                 421
```

Die Tabelle `hotels` enthält **6 Hotels**. Das neue Hotel AlpenStay Lugano (Eröffnung 2026) hat bereits Zimmer, aber noch keine Buchungen. Der Analyst ist verwirrt: *«Ich habe doch extra `LEFT JOIN` verwendet!»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 6 Hotels** erscheinen. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

Die `LEFT JOIN`s sind richtig: Sie erzeugen für AlpenStay Lugano Zeilen mit NULL in allen `bookings`-Spalten. Das Problem ist die Bedingung im **`WHERE`**: `WHERE` wird **nach** dem JOIN auf jede Zeile angewendet. Für Lugano ist `b.status` NULL, und `NULL = 'confirmed'` ist nicht wahr — die Zeilen werden entfernt. Damit wirkt der `LEFT JOIN` im Ergebnis wie ein `INNER JOIN`.

**Änderung:** Die Bedingung `b.status = 'confirmed'` gehört in die **`ON`-Klausel** des `LEFT JOIN` auf `bookings`. Dann werden nur bestätigte Buchungen angehängt; Hotels bzw. Zimmer ohne bestätigte Buchung bleiben trotzdem erhalten (mit NULL), und `COUNT(b.bookingid)` zählt für sie `0`.

Ergebnis: alle 6 Hotels, **AlpenStay Lugano mit 0** bestätigten Buchungen; die Zahlen der übrigen Hotels bleiben gleich (Bern 469 … Basel 421).

**Korrigierte SQL-Abfrage:**

In [5]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
SELECT
    h.name,
    COUNT(b.bookingid) AS confirmed_bookings
FROM hotels h
LEFT JOIN rooms r    ON h.hotelid = r.hotelid
LEFT JOIN bookings b ON r.roomid  = b.roomid
                    AND b.status  = 'confirmed'
GROUP BY h.name
ORDER BY confirmed_bookings DESC
""", con=engine)

engine.dispose()
df_B2

,name,confirmed_bookings
0,AlpenStay Bern,469
1,AlpenStay Grindelwald,462
2,AlpenStay Luzern,428
3,AlpenStay Zermatt,423
4,AlpenStay Basel,421
5,AlpenStay Lugano,0


> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Alternative: ohne Filter joinen und bedingt zählen, z.B. `SUM(CASE WHEN b.status = 'confirmed' THEN 1 ELSE 0 END)`. Eine Lösung mit `WHERE b.status = 'confirmed' OR b.status IS NULL` liefert hier zufällig ebenfalls 6 Zeilen, würde aber ein Hotel mit ausschliesslich stornierten Buchungen weiterhin verlieren.*
>
> *Bezug: Folien Woche 1, S. 40 (Joins: LEFT JOIN); AP01 B1 (LEFT JOIN, `COUNT(t.ticketid)`).*

---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `hotel.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Die gültigen Werte in `guests.loyalty_level` sind `Basic`, `Silver` und `Gold`. Schreiben Sie eine SQL-Abfrage, die alle Gäste (`guestid`, `firstname`, `lastname`, `loyalty_level`) mit einem ungültigen **oder fehlenden** Wert in `loyalty_level` ausgibt, sortiert nach `guestid`. Begründen Sie Ihre Filterbedingung.

**Begründung der Vorgehensweise:**

`NOT IN ('Basic', 'Silver', 'Gold')` findet abweichende Schreibweisen (`gold`, `GOLD`, `basic`), Tippfehler bzw. deutsche Varianten (`Silber`) und nicht vorgesehene Stufen (`Platin`). Zeilen mit NULL liefert `NOT IN` aber **nicht**: Der Vergleich mit NULL ergibt «unbekannt» (vgl. A3). Deshalb braucht es zusätzlich `OR loyalty_level IS NULL`.

Ergebnis: **29 Gäste** — 5 mit ungültigem Wert und 24 ohne Wert.

**SQL-Abfrage:**

In [6]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
SELECT
    guestid,
    firstname,
    lastname,
    loyalty_level
FROM guests
WHERE loyalty_level NOT IN ('Basic', 'Silver', 'Gold')
   OR loyalty_level IS NULL
ORDER BY guestid
""", con=engine)

engine.dispose()
df_C1

,guestid,firstname,lastname,loyalty_level
0,G0008,Sara,Weber,NaN
1,G0023,Noah,Baumann,NaN
2,G0109,Mia,Gerber,gold
3,G0130,Matteo,Meier,basic
4,G0136,Anna,Rossi,NaN
5,G0181,Mia,Zimmermann,NaN
6,G0192,Lea,Frei,NaN
7,G0193,Sara,Rossi,GOLD
8,G0197,Julia,Gerber,NaN
9,G0232,Mia,Steiner,NaN


> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Hinweis: Ohne `OR loyalty_level IS NULL` liefert die Abfrage nur 5 Zeilen («falsche Filterung»).*
>
> *Bezug: AP01 C1 (`NOT IN` für ungültige Werte); Folien Woche 1, S. 43 (Data Quality: complete, consistent).*

### C2 (4 Punkte)

Wie lange bleiben die Gäste und wie viel bezahlen sie pro Zimmertyp? Geben Sie für **bestätigte** Buchungen pro `room_type` die Anzahl Buchungen (`bookings`), die durchschnittliche Anzahl Nächte (`avg_nights`) und den durchschnittlich bezahlten Preis pro Nacht (`avg_rate`) aus — beide gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`. Sortieren Sie absteigend nach `avg_rate`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

`room_type` liegt in `rooms`, `nights`, `rate_paid` und `status` in `bookings`; beide Tabellen sind über `roomid` verbunden. `WHERE b.status = 'confirmed'` schliesst stornierte Buchungen aus, `GROUP BY r.room_type` gruppiert nach Zimmertyp, `AVG()` berechnet die Durchschnitte. `CAST(... AS NUMERIC)` ist nötig, damit `ROUND(..., 2)` auch in PostgreSQL funktioniert.

Ergebnis: Suiten sind mit durchschnittlich CHF 494.50 pro Nacht am teuersten; die Aufenthaltsdauer ist bei allen Zimmertypen ähnlich (rund 3 Nächte).

**SQL-Abfrage:**

In [7]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
SELECT
    r.room_type,
    COUNT(b.bookingid) AS bookings,
    ROUND(CAST(AVG(b.nights) AS NUMERIC), 2)    AS avg_nights,
    ROUND(CAST(AVG(b.rate_paid) AS NUMERIC), 2) AS avg_rate
FROM rooms r
JOIN bookings b ON r.roomid = b.roomid
WHERE b.status = 'confirmed'
GROUP BY r.room_type
ORDER BY avg_rate DESC
""", con=engine)

engine.dispose()
df_C2

,room_type,bookings,avg_nights,avg_rate
0,Suite,279,3.03,494.50
1,Familienzimmer,389,3.15,284.93
2,Doppelzimmer,1182,3.00,236.69
3,Einzelzimmer,353,2.86,159.51


> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Bezug: AP01 C2 (`AVG`, `ROUND(CAST(... AS NUMERIC), 2)`); Folien Woche 1, S. 39–40.*

### C3 (4 Punkte)

Über welchen Buchungskanal wird am häufigsten storniert? Geben Sie pro Kanal `channel_name`, die Anzahl Buchungen (`bookings`), die Anzahl stornierter Buchungen (`cancelled`) und die Stornoquote in Prozent (`cancel_rate_pct`, gerundet auf 1 Dezimalstelle) aus, absteigend sortiert nach `cancel_rate_pct`. Begründen Sie, wie Sie die stornierten Buchungen zählen und worauf Sie bei der Division achten müssen.

**Begründung der Vorgehensweise:**

`channel_name` liegt in `channels`, `status` in `bookings` — JOIN über `channelid`. Die stornierten Buchungen werden mit `SUM(CASE WHEN b.status = 'cancelled' THEN 1 ELSE 0 END)` gezählt (jede stornierte Buchung zählt 1, alle anderen 0). Ein `WHERE status = 'cancelled'` ginge nicht, weil dann für die Gesamtzahl `bookings` die bestätigten Buchungen fehlen würden.

Division: `SUM(...)` und `COUNT(...)` sind ganze Zahlen; in PostgreSQL und SQLite ergibt `216 / 1079` deshalb **0** (Ganzzahldivision). Mit `100.0 * ...` wird zuerst in eine Dezimalzahl umgewandelt.

Ergebnis: Über die **Buchungsplattform** wird mit **20.0 %** mit Abstand am häufigsten storniert (216 von 1079 Buchungen); über die Website nur 5.2 %.

**SQL-Abfrage:**

In [8]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
SELECT
    c.channel_name,
    COUNT(b.bookingid) AS bookings,
    SUM(CASE WHEN b.status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled,
    ROUND(CAST(100.0 * SUM(CASE WHEN b.status = 'cancelled' THEN 1 ELSE 0 END)
               / COUNT(b.bookingid) AS NUMERIC), 1) AS cancel_rate_pct
FROM channels c
JOIN bookings b ON c.channelid = b.channelid
GROUP BY c.channel_name
ORDER BY cancel_rate_pct DESC
""", con=engine)

engine.dispose()
df_C3

,channel_name,bookings,cancelled,cancel_rate_pct
0,Buchungsplattform,1079,216,20.0
1,Reisebüro,264,19,7.2
2,Telefon,265,16,6.0
3,Website,892,46,5.2


> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Alternative für PostgreSQL: `COUNT(*) FILTER (WHERE b.status = 'cancelled')` (läuft nicht in jeder SQLite-Version). Wer `100 * SUM(...) / COUNT(...)` schreibt, erhält abgeschnittene ganze Zahlen (20, 7, 6, 5) → «fehlende Rundung» bzw. kleiner Fehler.*
>
> *Bezug: Folien Woche 1, S. 38 (Data Types: Ganzzahl vs. Dezimalzahl) und S. 39 (SQL-Abfrage).*

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [9]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.18.44-fc-v310
Datetime: 2026-11-07 10:47:26
Python Version: 3.13.18
IP Address: 127.0.0.1
-----------------------------------
